# AGO
---
## Requisitos
- Geração aleatória
- Seleção por proporcionalidade do fitness e por torneio 
- Cruzamento por 1 ponto e por 2 pontos - Podendo ajustar a taxa de cruzamento
- Mutação - Podendo ajustar a taxa de mutação
- Sphere: 30 dimensões (ou seja 30 variáveis) #parametrização
        - Os limites de cada variável é de: -100 a 100  #parametrização
- Tamanho da População = 30 
- Quantidade de gerações = 20 

---

- Cromossomo: É a estrutura de dados que representa uma solução completa para o problema. No código, cada indivíduo da lista pop será um cromossomo.
- Gene: É cada elemento individual dentro dessa estrutura (cada valor na lista). Se o problema tem 6 variáveis que precisam ser ajustadas para encontrar o melhor resultado, então o cromossomo terá 6 genes. Ou seja, na Sphere, seriam 6 dimensões

- Sobre os "var_x": são limites, definem o intervalo válido para o valor de cada gene. No caso da Sphere, cada variável (gene) deve estar entre -100 e 100: cromossomo = [45.3, -72.1, 0.5, 88.9, -12.4, ...]

In [934]:
import random

In [935]:
# parametros do AG
n_pop = 30
quantidade_genes = 30
n_geracoes = 20
pop = []
nova_pop = []
n_cortes = 2
taxa_crossover = 0.5242
taxa_mutacao = 0.0161
selecao='torneio'


# Parametros da sphere
n_dimensoes = quantidade_genes
var_min = -100
var_max = 100



## Gerar População

In [936]:
pop.clear()
#Loop para criar cada indivíduo da população:
for i in range(n_pop):
    cromossomo = [random.uniform(var_min, var_max) for _ in range(n_dimensoes)]
    pop.append(cromossomo)
    #print(f"cromossomo {i+1} gerado: {cromossomo}")
    #print("\n")

In [937]:
#pop

## Função Sphere

In [938]:
def sphere(x):
    """
    Sphere Function
    Mínimo global: f(0, ..., 0) = 0
    """
    return sum(xi**2 for xi in x)

## Avaliar cada individuo 

In [939]:
pop_aval= []
pop_aval.clear()
for cromossomo in pop:
    nota_aval = sphere(cromossomo)
    pop_aval.append((cromossomo, nota_aval))

#pop_aval
    

## Selecionar os individuos mais aptos

- Pais 1 com ROLETA
- lembrando: problema de minimização -> quanto menor a avaliação, maior o valor da aptidão.
- fitness = 1/(1 + nota avaliaçao)
- pop_aval = [(cromossomo, nota_aval), ...]

In [940]:
def calcular_fitness(pop_avaliada):
    """
    Recebe: lista de tuplas (cromossomo, valor_sphere)
    Retorna: lista de pesos de fitness
    """
    return [1 / (1 + individuo[1]) for individuo in pop_avaliada]

#def mostrar_probabilidade(pop_avaliada):
#    """
#    Mostra cada cromossomo com sua avaliação e probabilidade relativa
#    """
#    fitness = calcular_fitness(pop_avaliada)
#    total = sum(fitness)
#    probabilidades = [f/total for f in fitness]
#
#    for i, (crom, fitness) in enumerate(pop_avaliada):
#        print(f"Cromossomo {i+1}: avaliação={fitness:.2f}, probabilidade={probabilidades[i]*100:.2f}%")

def selecao_proporcional(pop_avaliada, n_pop_selec):
    """
    Recebe: população avaliada e quantidade a selecionar
    Retorna: apenas os cromossomos selecionados (sem o valor da sphere)
    """
    fitness = calcular_fitness(pop_avaliada)

    grupo_survivors = random.choices(pop_avaliada, weights=fitness, k=n_pop_selec)
    
    # retorna só os cromossomos, sem o valor da sphere
    return [individuo[0] for individuo in grupo_survivors]

In [941]:
#mostrar_probabilidade(pop_aval)
#len(p1_survivors)

In [942]:
def selecao_torneio(pop_avaliada, n_pop_selec):
    """
    Recebe: população avaliada [(cromossomo, avaliação), ...]
    Retorna: lista de cromossomos selecionados via torneio
    """
    grupo_survivors = []
    for rodada in range(n_pop_selec):
        # sorteiar 2 competidores
        c1, c2 = random.choices(pop_avaliada, k=2)

        # mostrar os competidores
        #print(f"Torneio {rodada+1}:")
        #print(f"  Competidor 1 -> avaliação={c1[1]:.2f}")
        #print(f"  Competidor 2 -> avaliação={c2[1]:.2f}")

        # comparar avaliações (menor é melhor)
        if c1[1] < c2[1]:
            survivor = c1
        else:
            survivor = c2

        #print(f"  Vencedor -> avaliação={survivor[1]:.2f}\n")

        grupo_survivors.append(survivor[0])  # pega só o cromossomo

    return grupo_survivors

In [943]:
if selecao == 'torneio':
    p1_survivors = selecao_torneio(pop_aval, n_pop)
    p2_survivors= selecao_torneio(pop_aval, n_pop)
else:
    p1_survivors = selecao_proporcional(pop_aval, n_pop)
    p2_survivors = selecao_proporcional(pop_aval, n_pop)

len(p2_survivors)

30

In [944]:
def crossover_1pt(p1, p2, taxa_crossover, quantidade_genes):
    """
    Recebe: dois pais (Selecionados/sobreviventes), taxa de crossover e a quantidade de genes
    Retorna: dois filhos
    """
    # verifica se há cruzamento
    if random.random() < taxa_crossover:
        ponto_corte = random.randint(1, quantidade_genes - 1)
        
        # troca as partes após o ponto de corte
        filho1 = p1[:ponto_corte] + p2[ponto_corte:]
        filho2 = p2[:ponto_corte] + p1[ponto_corte:]
        #print(f"crossover: {filho1} + {filho2}\n")
    else:
        # sem cruzamento, filhos são cópias dos pais
        filho1 = p1[:]
        filho2 = p2[:]
        #print(f"sem crossover: {filho1} + {filho2}\n")
    
    return filho1, filho2

In [945]:
def mutacao(cromossomo, taxa_mutacao, quantidade_genes, var_min, var_max):
    """
    Recebe: cromossomo, quantidade de genes, taxa de mutação e limites
    Retorna: cromossomo (com possíveis mutações)
    """
    cromossomo_mutado = cromossomo[:]  # copia para não alterar o original
    
    for i in range(quantidade_genes):
        if random.random() < taxa_mutacao:
            cromossomo_mutado[i] = random.uniform(var_min, var_max)
            #print(f"mutacao no {i+1}: {cromossomo} ---> {cromossomo_mutado}\n")
    
    return cromossomo_mutado

In [946]:
def gerar_nova_pop(p1_survivors, p2_survivors, quantidade_genes,n_pop,
                          taxa_crossover, taxa_mutacao, 
                          var_min, var_max, nova_pop):
    """
    Recebe: listas de pais selecionados, quantidade de genes, a lista de nova populção, taxas e limites
    Retorna: nova população após crossover e mutação
    """
    
    # percorre os pais em pares
    for i in range(0, n_pop):
        
        p1 = random.choice(p1_survivors)
        p2 = random.choice(p2_survivors)
        
        # crossover
        filho1, filho2 = crossover_1pt(p1, p2, taxa_crossover, quantidade_genes)
        
        # mutação em cada filho
        filho1 = mutacao(filho1, taxa_mutacao, quantidade_genes, var_min, var_max)
        filho2 = mutacao(filho2, taxa_mutacao, quantidade_genes, var_min, var_max)
        
        nova_pop.append(filho1)
        nova_pop.append(filho2)
        
    return nova_pop

In [947]:
nova_pop.clear()
gerar_nova_pop(p1_survivors, p2_survivors, quantidade_genes, n_pop, taxa_crossover, taxa_mutacao, var_min, var_max, nova_pop)

[[-28.448974000836813,
  -31.797653235345024,
  -70.53188695974072,
  -18.263623879532247,
  98.01572406124447,
  71.04652404782948,
  -47.260749397675774,
  65.63444018831393,
  -16.375143023855117,
  48.74760036187078,
  -30.61495415450844,
  33.71512722467327,
  -5.651092719186224,
  57.245050966030306,
  -28.319669961581724,
  -24.71292297239762,
  68.99145295081252,
  -62.08875633066502,
  88.57049687981132,
  92.4856555063783,
  91.43860041130031,
  -48.99015600658778,
  -15.76244993213787,
  44.68356906505676,
  71.55011803729965,
  -53.459803687572325,
  6.079558616936524,
  17.355213146443276,
  46.08289801104783,
  -41.66262737302959],
 [-18.43993695031341,
  -91.12526366559703,
  -54.18039084222108,
  28.966172520371146,
  68.10152096776997,
  -27.89145679194587,
  10.923293289382173,
  85.30551479244897,
  90.72730246515829,
  -70.27151823374125,
  49.45547779327799,
  -12.271050111087334,
  32.131530741773446,
  -26.044695903655878,
  -43.10700316796765,
  -20.223286320238

In [948]:
#len(nova_pop)

In [949]:
#print(nova_pop)

## Loop exec - Critério de parada - GERAÇÕES

In [950]:
historico_melhor = []  # para visualizar a evolução

for geracao in range(n_geracoes):

    # 1. Avaliar população atual
    pop_aval = [(crom, sphere(crom)) for crom in pop]

    # 2. Selecionar pais
    p1_survivors = selecao_proporcional(pop_aval, n_pop)
    p2_survivors = selecao_torneio(pop_aval, n_pop)

    # 3. Gerar 60 filhos
    nova_pop.clear()
    gerar_nova_pop(p1_survivors, p2_survivors, quantidade_genes, n_pop,
                   taxa_crossover, taxa_mutacao, var_min, var_max, nova_pop)

    # 4. Avaliar os 60 filhos e pegar os 30 melhores
    filhos_aval = [(crom, sphere(crom)) for crom in nova_pop]
    filhos_aval.sort(key=lambda x: x[1])          # ordena do menor para o maior
    pop = [ind[0] for ind in filhos_aval[:n_pop]]  # pega os 30 melhores

    # 5. Registrar o melhor dessa geração
    melhor_fitness = filhos_aval[0][1]
    historico_melhor.append(melhor_fitness)
    print(f"Geração {geracao+1:02d} | Melhor fitness: {melhor_fitness:.2f}")

# Resultado final
print(f"\n{'='*40}")
print(f"RESULTADO FINAL após {n_geracoes} gerações:")
print(f"Melhor fitness encontrado: {min(historico_melhor):.2f}")
print(f"Geração onde ocorreu: {historico_melhor.index(min(historico_melhor)) + 1}")

Geração 01 | Melhor fitness: 39522.29
Geração 02 | Melhor fitness: 39522.29
Geração 03 | Melhor fitness: 39272.82
Geração 04 | Melhor fitness: 39522.29
Geração 05 | Melhor fitness: 38108.31
Geração 06 | Melhor fitness: 37467.45
Geração 07 | Melhor fitness: 35460.51
Geração 08 | Melhor fitness: 29868.58
Geração 09 | Melhor fitness: 28482.10
Geração 10 | Melhor fitness: 18085.03
Geração 11 | Melhor fitness: 18085.03
Geração 12 | Melhor fitness: 18085.03
Geração 13 | Melhor fitness: 16837.86
Geração 14 | Melhor fitness: 15211.27
Geração 15 | Melhor fitness: 14795.56
Geração 16 | Melhor fitness: 14273.10
Geração 17 | Melhor fitness: 14667.97
Geração 18 | Melhor fitness: 11386.43
Geração 19 | Melhor fitness: 11386.43
Geração 20 | Melhor fitness: 11335.06

RESULTADO FINAL após 20 gerações:
Melhor fitness encontrado: 11335.06
Geração onde ocorreu: 20
